**Volume 1 workbooks v0.1.0, published 2026-08-19.**


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

try:
    np.set_printoptions(legacy='1.25')
except TypeError:
    pass
np.set_printoptions(suppress=True)
pd.set_option('display.float_format', '{:,.10g}'.format)

import pandas as pd

from ada_fsaudit_bridge import (
    att_sample,
    configure_environment,
    cvs_sample,
    load_dataset,
    lower_bound,
    mus_sample,
    set_notebook_context,
    upper_bound,
)
from scipy.stats import hypergeom

ADA_WORKSHOP_ID = "auxiliary-variables-and-stratification"
ADA_CHAPTER = "3"
ADA_NOTEBOOK_SOURCE = "notebooks/support/auxiliary-variables-and-stratification/support.Rmd"

def ada_set_context(exercise_ref: str) -> None:
    set_notebook_context(
        chapter=ADA_CHAPTER,
        exercise=exercise_ref,
        notebook=f'{ADA_WORKSHOP_ID}:{ADA_NOTEBOOK_SOURCE}',
    )

configure_environment()


## Exercise 3.1. The `inventoryData` file


Details of the data file can be obtained by inspecting the dataset columns and
row count.


In [ ]:
ada_set_context("3.1")
inventoryData = load_dataset("inventoryData")
inventoryData.columns.tolist()
len(inventoryData)
display(inventoryData["bv"].sum())
display(inventoryData.iloc[:6, :3])


Use these outputs as a data-readiness check. Later calculations rely on these records as the population evidence base, so a mismatch in variables or population size would affect every estimate that follows.


## Exercise 3.2. Working with the CVS object


We first create a Classical Variables Sampling ("CVS") object. An object is a
container that gathers all the data relevant to a particular sampling
application. The Python bridge represents this CVS object through the helper
`cvs_sample(...)`; Chapter 4 uses analogous helpers for attribute and MUS
sampling.
The practical goal is to keep the sampling plan, selected items, audit values,
and evaluation results together so each later estimator is applied to the same
evidence. We create the CVS object `mySample` with `cvs_sample(...)`, and at the
same time fill it with information on the required sample size `n`, the relevant
book values `bv`, identifiers `id`, and the random seed number `seed`.


In [ ]:
ada_set_context("3.2")
mySample = cvs_sample(
    n=400,
    bv=inventoryData["bv"],
    id=inventoryData["item"],
    seed=12345,
)


The CVS object has 19 different attributes attached to it, that will be filled as we proceed from sample planning, through stratification and selection to evaluation. The following is a list of all attributes.


In [ ]:
ada_set_context("3.2")
mySample.field_names()


Some of these attributes are empty for now, for example `sample`.


In [ ]:
ada_set_context("3.2")
mySample.sample


The selection in this workshop runs through the FSaudit package in R, via the
notebook bridge. Because the bridge draws in the same R engine with the same
seed and the same `Rounding` sample kind as the R workshop, the Python
workflow selects exactly the same items and produces the same estimates as
the R workshop and the book.


In [ ]:
ada_set_context("3.2")
configure_environment(seed=12345, r_sample_kind="Rounding")


We sample 400 units from the `inventoryData` dataset with the `select()` function. This effectively fills the `sample` attribute in `mySample`.


In [ ]:
ada_set_context("3.2")
mySample.select()
mySample.sample.head()


The selected rows identify the inventory items that would be audited. We now look up the corresponding audit values and submit them to the CVS object for evaluation.


In [ ]:
ada_set_context("3.2")
audit_values = inventoryData.set_index("item").loc[mySample.sample["item"], "av"]
mySample.evaluate(av=audit_values)


The evaluation results are stored in the `evalResults` attribute of the CVS object. This attribute itself is a list of 17 different attributes. The next output is a map of where the estimates, precision measures, and supporting details are stored for later interpretation.


In [ ]:
ada_set_context("3.2")
list(mySample.eval_results.keys())


## Exercise 3.3. Mean-per-unit estimator


The mean-per-unit estimator is the baseline for this chapter. We first retrieve the estimated population audit value and achieved precision so later auxiliary-variable estimators can be judged against the same sample evidence.


In [ ]:
ada_set_context("3.3")
mySample.eval_results["Most likely total audited amount mpu"]


In [ ]:
ada_set_context("3.3")
mySample.eval_results["Achieved precision mpu"]


Read the estimate together with the achieved precision. A wider precision value means more uncertainty remains around the estimated population value, even when the point estimate itself looks plausible.
The prediction interval turns that uncertainty into lower and upper bounds:


In [ ]:
ada_set_context("3.3")
mySample.eval_results["Estimates"].iloc[3:5, 0]


In this call, the numbers in brackets refer to the 4th and 5th row of the Estimates table. The first column stores the results of the mean-per-unit estimator. Before moving to other estimators, note the baseline interval width; this is what auxiliary variables should improve if they are useful in this audit setting.
Alternatively, we can refer to the label of the relevant column.


In [ ]:
ada_set_context("3.3")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("mpu")]


## Exercise 3.4. The Regression Estimator


This exercise reviews the regression estimator for the same sample evidence. The practical question is whether the auxiliary relationship between book value and audit value reduces uncertainty compared with the mean-per-unit baseline.
The results for the regression estimator are stored in a similar way.


In [ ]:
ada_set_context("3.4")
mySample.eval_results["Most likely total audited amount regression"]


In [ ]:
ada_set_context("3.4")
mySample.eval_results["Achieved precision regression"]


Interpret the point estimate and achieved precision together. A useful estimator is not only close to the expected audited amount; it should also give a narrower uncertainty range for the audit conclusion.
The prediction interval is determined by:


In [ ]:
ada_set_context("3.4")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("regr")]


Read the lower and upper bounds as the range of audited inventory values supported by this regression estimate.


## Exercise 3.5. The Difference Estimator


This exercise evaluates the difference estimator. Here the audit task is to estimate the total effect of differences between book values and audit values, then compare that adjustment with the other auxiliary-variable methods.
The results for the difference estimator are as follows.


In [ ]:
ada_set_context("3.5")
mySample.eval_results["Most likely total audited amount difference"]


In [ ]:
ada_set_context("3.5")
mySample.eval_results["Achieved precision difference"]


Use the achieved precision to judge whether the difference adjustment gives enough practical benefit over the baseline estimate.
The prediction interval is determined by:


In [ ]:
ada_set_context("3.5")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("diff")]


Before moving on, compare these bounds with the regression bounds. The stronger method for this sample is the one that gives a defensible estimate with less remaining uncertainty.


## Exercise 3.6. The Ratio Estimator


Finally, we retrieve the ratio estimator results. Like the regression and difference estimators, the ratio estimator uses the relationship between book value and audit value to try to reduce uncertainty.


In [ ]:
ada_set_context("3.6")
mySample.eval_results["Most likely total audited amount ratio"]


In [ ]:
ada_set_context("3.6")
mySample.eval_results["Achieved precision ratio"]


The prediction interval is determined by:


In [ ]:
ada_set_context("3.6")
mySample.eval_results["Estimates"].iloc[3:4 + 1, mySample.eval_results["Estimates"].columns.get_loc("ratio")]


An overview of all estimates is given by


In [ ]:
ada_set_context("3.6")
mySample.eval_results["Estimates"]


Use this table to compare methods on the same evidence base. Focus first on the precision row and the lower and upper bounds: the auxiliary-variable estimators should be evaluated by how much uncertainty they remove relative to the mean-per-unit baseline, not only by which point estimate is largest or smallest.


## Exercise 3.7. Using CVS with sporadic errors


This exercise switches from inventories to accounts receivable and uses CVS when
errors are sporadic. The practical task is to select the sample, attach the
audited values, and identify whether the sporadic-error calculation is
triggered.
We start with setting up a CVS sample object with the Python bridge helper
`cvs_sample(...)`, creating the object named `arSample`, and fill it with the parameters. We use the prefix `ar` for
accounts receivable.


In [ ]:
ada_set_context("3.7")
accounts_receivable = load_dataset("accounts_receivable")
arSample = cvs_sample(
    bv=accounts_receivable["amount"],
    id=accounts_receivable["invoice"],
    n=100,
)


We select the sample, using a seed equal to 1.


In [ ]:
ada_set_context("3.7")
arSample.select(seed=1)


We look up the audit values of the selected sampling units. This step connects the planned sample to the fieldwork evidence that will be evaluated.


In [ ]:
ada_set_context("3.7")
audit_values = accounts_receivable.set_index("invoice").loc[arSample.sample["item"], "av2"]


We then evaluate the sample and look at the `Estimates` attribute.


In [ ]:
ada_set_context("3.7")
arSample.evaluate(av=audit_values)
arSample.eval_results["Estimates"]


The number of differences is stored in the `#_Errors` attribute, under each of the estimation methods. This way, you can tell if precision was calculated with the sporadic error (k = 3 to 19) method, or using the standard CVS method.


In [ ]:
ada_set_context("3.7")
arSample.eval_results["Regression estimation"]["#_Errors"]


Use the number of errors as a method check before interpreting the interval. The audit conclusion should be based on the evaluation method that matches the observed error pattern.
Most of the calculations presented in the manuscript section on using CVS with sporadic errors develop the full calculations step by step; here we highlight the two ingredients that are new compared with the unstratified case: the upper bound $M_U$ on the proportion of errors in the population, and the use of $m - 1$ degrees of freedom for the $t$-value.
The value of $M_U$ that is used in, for example, the sporadic-difference standard-deviation equation is obtained as follows.


In [ ]:
ada_set_context("3.7")
N = len(accounts_receivable)
m = 6
(m_u := upper_bound(k=m, popn=N, n=100, alpha=0.05) / N)


The effective degrees of freedom is $m - 1$. This explains the $t$-value used when calculating the confidence intervals.


In [ ]:
from scipy.stats import t
ada_set_context("3.7")
t.ppf(0.975, df=m - 1)


Treat these values as ingredients in the interval calculation rather than separate audit conclusions.


## Exercise 3.8. Stratification with equal recorded boundaries


This exercise creates a first stratified design using equal recorded-value boundaries. The practical question is whether this simple rule divides the inventory population into useful groups before sample sizes are assigned.
We start with creating a CVS object named `equal`.


In [ ]:
ada_set_context("3.8")
equal = cvs_sample(bv=inventoryData["bv"], id=inventoryData["item"])


We then stratify with the equal method (`stratMeth = equal`), and view the summary.


In [ ]:
ada_set_context("3.8")
equal.stratify(strata=3, stratMeth="equal")
equal.field("stratSumm")


Inspect the stratum summary before comparing methods. If most book value is concentrated in only one stratum, equal boundaries may be less helpful than a stratification rule designed around the cumulative value distribution. Once you have reproduced the summary, experiment with a larger number of strata (for example 5 or 10) to see how the boundaries and the number of items per stratum change.


## Exercise 3.9. Stratification with the cumulative method


This exercise applies stratification before sample selection. The practical question is whether grouping inventory items by book value can produce a sampling design that reaches the desired precision more efficiently.
We create a new CVS object named `cumul`, and use the stratification with the cumulative method rather than the equal recorded boundaries of Exercise 3.8: because the book values are strongly right-skewed, equal book-value strata concentrate almost all items in the first stratum, whereas the cumulative square root of frequencies method balances the number of items and the variability per stratum.


In [ ]:
ada_set_context("3.9")
cumul = cvs_sample(bv=inventoryData["bv"], id=inventoryData["item"])
cumul.stratify(strata=3, classes=10, stratMeth="cumulative")


The classification in the manuscript inventory-classification table is stored in the `classSumm` attribute.


In [ ]:
ada_set_context("3.9")
cumul.field("classSumm")


The stratification summary is then:


In [ ]:
ada_set_context("3.9")
cumul.field("stratSumm")


We calculate the required sample size for a desired precision of 200,000. Read the resulting sample size as a planning output: it tells us how much evidence this stratified design expects to need before audit values are collected. Once you have reproduced the calculation, experiment with other values of `desPrec` to see how sensitive the required sample size is to the target precision.


In [ ]:
ada_set_context("3.9")
cumul.size(desPrec=200000).n


The calculated sample size must be allocated to the strata before the fieldwork sample can be selected. Here we use proportional allocation: each stratum receives a share of the sample based on its share of population items. Inspect the allocation and explain which stratum receives the most audit effort and why that follows from the cumulative stratification summary.


In [ ]:
ada_set_context("3.9")
import numpy as np
import pandas as pd
summ = cumul.field("stratSumm")
n = int(np.asarray(cumul.n).item())
alloc = pd.DataFrame({
    "stratum": summ["strat"],
    "items": summ["freq"],
})
alloc["share"] = alloc["items"] / alloc["items"].sum()
raw = n * alloc["share"]
alloc["planned_n"] = np.floor(raw).astype(int)
remaining_items = n - int(alloc["planned_n"].sum())
if remaining_items > 0:
    largest_remainders = (
        raw - np.floor(raw)
    ).sort_values(ascending=False).index
    alloc.loc[
        largest_remainders[:remaining_items],
        "planned_n",
    ] += 1
alloc


We then select the stratified sample with seed 12345. The selection runs
through the bridge in R, so the Python workflow draws exactly the same
stratified sample as the R workshop and the book.


In [ ]:
ada_set_context("3.9")
cumul.select(seed=12345)
cumul.sample.head()


The selected items define the fieldwork target for this stratified design. We then obtain the audit values and evaluate the sample.


In [ ]:
ada_set_context("3.9")
true_values = inventoryData.set_index("item").loc[cumul.sample["item"], "av"]
cumul.evaluate(av=true_values)


A summary table of the estimates for each of the estimation methods is stored in the `Estimates` argument, which is part of the `evalResults` argument.


In [ ]:
ada_set_context("3.9")
cumul.eval_results["Estimates"]


Compare this summary with the earlier unstratified results: in the manuscript, stratification brings the achieved precision of the MPU estimate down from 684,415 to 245,704, and the auxiliary-variable estimators to roughly 127,000, below the target precision of 200,000. The learning point is to interpret stratification as a design choice: it is useful when the reduction in uncertainty justifies the extra planning and implementation effort.
